In [ ]:
# -*- coding: utf-8 -*-
"""
911 TF-IDF + Sentiment:
- Train category model (EMS / Fire / Traffic)
- Train subcategory model (rare -> OTHER to avoid stratify errors)
- Analyze long English call transcripts:
  * predicted category + subcategory
  * sentiment (polarity, subjectivity)
  * extracted requirements (keywords + regex)
  * urgency score + dispatch suggestion

Install:
  python3.12 -m pip install numpy pandas scikit-learn textblob joblib
  python3.12 -m textblob.download_corpora

Run:
  python TextAnalize.py
"""

import re
import json
from dataclasses import dataclass
from typing import Dict, Any, List, Tuple, Optional

import pandas as pd
from textblob import TextBlob

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


# -----------------------------
# Text Preprocess
# -----------------------------
def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s).lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s


def build_labels_from_title(title: str) -> Tuple[str, str]:
    """
    Dataset title usually: "EMS: CARDIAC EMERGENCY"
    returns: (category, subcategory)
    """
    if not isinstance(title, str):
        return "Unknown", "Unknown"

    parts = title.split(":", 1)
    if len(parts) == 2:
        cat = parts[0].strip()
        sub = parts[1].strip()
        return (cat if cat else "Unknown"), (sub if sub else "Unknown")

    return "Unknown", title.strip() if title.strip() else "Unknown"


# -----------------------------
# Sentiment (English)
# -----------------------------
def sentiment_textblob(text: str) -> Dict[str, float]:
    tb = TextBlob(text)
    return {
        "polarity": float(tb.sentiment.polarity),       # [-1, 1]
        "subjectivity": float(tb.sentiment.subjectivity) # [0, 1]
    }


# -----------------------------
# Requirement / Need Extraction (English only)
# -----------------------------
KEYWORD_BUCKETS: Dict[str, List[str]] = {
    "life_threatening": [
        "not breathing", "no pulse", "unconscious", "cardiac arrest", "heart attack",
        "severe bleeding", "bleeding heavily", "choking", "stroke", "seizure", "overdose",
        "cannot breathe", "difficulty breathing"
    ],
    "fire_hazard": [
        "fire", "smoke", "gas leak", "explosion", "burning"
    ],
    "violence_weapon": [
        "gun", "knife", "weapon", "shots fired", "shooting", "assault", "stabbing"
    ],
    "traffic_accident": [
        "car crash", "accident", "vehicle", "hit and run", "rollover", "collision"
    ],
    "medical": [
        "chest pain", "difficulty breathing", "diabetic", "allergic", "pregnant", "fever",
        "broken bone", "fracture"
    ],
    "police_need": [
        "break in", "robbery", "burglary", "threat", "domestic violence", "trespassing"
    ],
}

LOCATION_PATTERNS = [
    # "123 Main Street", "45 Elm Rd"
    r"\b\d{1,5}\s+[A-Za-z]+(?:\s+[A-Za-z]+)*\s+"
    r"(Street|St|Road|Rd|Avenue|Ave|Boulevard|Blvd|Lane|Ln|Drive|Dr|Court|Ct|Way|Highway|Hwy)\b",

    # "on I-95", "at Central Park", "in Downtown"
    r"\b(at|in|on)\s+[A-Za-z0-9\s\-\.\,]{3,}\b",

    # Highways / routes
    r"\b(I-\d+|US\s?\d+|Route\s?\d+|Highway\s?\d+)\b",

    # ZIP code
    r"\b\d{5}(?:-\d{4})?\b"
]

NUMBER_OF_PEOPLE_PATTERNS = [
    r"\b(\d+)\s+(people|persons|victims|injured)\b",
    r"\b(one|two|three|four|five|six|seven|eight|nine|ten)\s+(people|persons|victims|injured)\b",
]


def extract_requirements(text: str) -> Dict[str, Any]:
    t = normalize_text(text)
    found = {k: [] for k in KEYWORD_BUCKETS.keys()}

    for bucket, kws in KEYWORD_BUCKETS.items():
        for kw in kws:
            if kw in t:
                found[bucket].append(kw)

    locations: List[str] = []
    for pat in LOCATION_PATTERNS:
        for m in re.finditer(pat, text, flags=re.IGNORECASE):
            locations.append(m.group(0).strip())

    people_mentions: List[str] = []
    for pat in NUMBER_OF_PEOPLE_PATTERNS:
        for m in re.finditer(pat, text, flags=re.IGNORECASE):
            people_mentions.append(m.group(0).strip())

    # Deduplicate
    locations = list(dict.fromkeys([x for x in locations if len(x) >= 3]))[:10]
    people_mentions = list(dict.fromkeys(people_mentions))[:10]

    return {
        "signals": found,
        "possible_locations": locations,
        "people_mentions": people_mentions,
    }


# -----------------------------
# Urgency scoring (0..1)
# -----------------------------
CATEGORY_BASE_URGENCY = {
    "EMS": 0.65,
    "Fire": 0.75,
    "Traffic": 0.55,
    "Unknown": 0.40,
}

SUBCATEGORY_URGENCY_HINTS = {
    "CARDIAC": 0.25,
    "HEART": 0.25,
    "UNCONSCIOUS": 0.25,
    "BREATH": 0.20,
    "FIRE": 0.20,
    "GAS": 0.20,
    "EXPLOSION": 0.25,
    "SHOTS": 0.25,
    "SHOOT": 0.25,
    "ASSAULT": 0.20,
    "STABB": 0.25,  # stabbing/stabbed
}


def compute_urgency_score(
    predicted_category: str,
    predicted_subcategory: str,
    sentiment: Dict[str, float],
    reqs: Dict[str, Any],
) -> Dict[str, Any]:
    cat = predicted_category if predicted_category in CATEGORY_BASE_URGENCY else "Unknown"
    score = CATEGORY_BASE_URGENCY[cat]

    sub_upper = (predicted_subcategory or "").upper()
    for key, add in SUBCATEGORY_URGENCY_HINTS.items():
        if key in sub_upper:
            score += add

    # keyword signals
    if reqs["signals"]["life_threatening"]:
        score += 0.30
    if reqs["signals"]["fire_hazard"]:
        score += 0.20
    if reqs["signals"]["violence_weapon"]:
        score += 0.20
    if reqs["signals"]["traffic_accident"]:
        score += 0.10

    # sentiment (lightweight contribution)
    pol = sentiment.get("polarity", 0.0)
    subj = sentiment.get("subjectivity", 0.0)
    if pol < -0.2:
        score += 0.05
    if subj > 0.6:
        score += 0.03

    score = max(0.0, min(1.0, score))

    if score >= 0.85:
        level = "CRITICAL"
    elif score >= 0.70:
        level = "HIGH"
    elif score >= 0.50:
        level = "MEDIUM"
    else:
        level = "LOW"

    return {"urgency_score": score, "urgency_level": level}


# -----------------------------
# Models
# -----------------------------
@dataclass
class EmergencyModels:
    category_model: Pipeline
    subcategory_model: Pipeline
    vectorizer_params: Dict[str, Any]
    subcategory_min_count: int


def _make_pipeline(vec_params: Dict[str, Any]) -> Pipeline:
    return Pipeline([
        ("tfidf", TfidfVectorizer(**vec_params)),
        ("clf", LogisticRegression(
            max_iter=2000,
            n_jobs=-1,
            class_weight="balanced",
            solver="lbfgs",
            multi_class="auto",
        )),
    ])


def train_models_from_911_csv(
    csv_path: str,
    max_rows: int = 200000,
    subcategory_min_count: int = 5,  # <-- IMPORTANT: avoids stratify error
) -> Tuple[EmergencyModels, Dict[str, Any]]:
    df = pd.read_csv(csv_path)

    if max_rows and len(df) > max_rows:
        df = df.sample(max_rows, random_state=42)

    # Ensure columns exist
    for col in ["desc", "title"]:
        if col not in df.columns:
            raise RuntimeError(f"Missing column in CSV: {col}")

    df["desc"] = df["desc"].astype(str)
    df["title"] = df["title"].astype(str)
    df["text"] = (df["title"] + " || " + df["desc"]).map(normalize_text)

    cats, subs = zip(*df["title"].map(build_labels_from_title))
    df["category"] = list(cats)
    df["subcategory"] = list(subs)

    # ---- subcategory fix: rare -> OTHER ----
    sub_counts = df["subcategory"].value_counts()
    rare_subs = sub_counts[sub_counts < subcategory_min_count].index
    df["subcategory_clean"] = df["subcategory"].where(~df["subcategory"].isin(rare_subs), "OTHER")

    vec_params = dict(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        max_features=120000,
    )

    category_model = _make_pipeline(vec_params)
    subcategory_model = _make_pipeline(vec_params)

    # Category split (usually safe)
    X_train, X_test, ycat_train, ycat_test = train_test_split(
        df["text"], df["category"],
        test_size=0.2, random_state=42, stratify=df["category"]
    )

    # Subcategory split (now safe because rare -> OTHER)
    Xs_train, Xs_test, ysub_train, ysub_test = train_test_split(
        df["text"], df["subcategory_clean"],
        test_size=0.2, random_state=42, stratify=df["subcategory_clean"]
    )

    category_model.fit(X_train, ycat_train)
    subcategory_model.fit(Xs_train, ysub_train)

    cat_pred = category_model.predict(X_test)
    sub_pred = subcategory_model.predict(Xs_test)

    report = {
        "python_note": "subcategory_clean uses rare->OTHER to allow stratified split",
        "train_rows_category": int(len(X_train)),
        "test_rows_category": int(len(X_test)),
        "train_rows_subcategory": int(len(Xs_train)),
        "test_rows_subcategory": int(len(Xs_test)),
        "classes_category": sorted(set(df["category"])),
        "classes_subcategory_clean_count": int(df["subcategory_clean"].nunique()),
        "category_report": classification_report(ycat_test, cat_pred, output_dict=False),
        "subcategory_report": classification_report(ysub_test, sub_pred, output_dict=False),
    }

    models = EmergencyModels(
        category_model=category_model,
        subcategory_model=subcategory_model,
        vectorizer_params=vec_params,
        subcategory_min_count=subcategory_min_count,
    )
    return models, report


# -----------------------------
# Inference
# -----------------------------
def analyze_call_text(models: EmergencyModels, call_transcript: str) -> Dict[str, Any]:
    text_norm = normalize_text(call_transcript)

    pred_cat = models.category_model.predict([text_norm])[0]
    pred_sub = models.subcategory_model.predict([text_norm])[0]  # will include OTHER sometimes

    sent = sentiment_textblob(call_transcript)
    reqs = extract_requirements(call_transcript)
    urg = compute_urgency_score(pred_cat, pred_sub, sent, reqs)

    # Simple dispatch suggestions
    dispatch: List[str] = []
    if urg["urgency_level"] in ("HIGH", "CRITICAL"):
        dispatch.append("priority_dispatch")

    if reqs["signals"]["fire_hazard"] or pred_cat == "Fire":
        dispatch.append("fire_department")
    if reqs["signals"]["life_threatening"] or pred_cat == "EMS":
        dispatch.append("ambulance")
    if reqs["signals"]["violence_weapon"] or reqs["signals"]["police_need"]:
        dispatch.append("police")

    dispatch = list(dict.fromkeys(dispatch))

    return {
        "predicted_category": pred_cat,
        "predicted_subcategory": pred_sub,
        "sentiment": sent,
        "requirements": reqs,
        "urgency": urg,
        "dispatch_suggestion": dispatch,
    }

def write_output_to_txt(
    file_path: str,
    train_report: Dict[str, Any],
    inference_result: Dict[str, Any],
):
    with open(file_path, "w", encoding="utf-8") as f:
        f.write("===== 911 NLP ANALYSIS OUTPUT =====\n\n")

        # ---- Training reports ----
        f.write("=== TRAIN REPORT (CATEGORY) ===\n")
        f.write(train_report["category_report"])
        f.write("\n\n")

        f.write("=== TRAIN REPORT (SUBCATEGORY - rare -> OTHER) ===\n")
        f.write(train_report["subcategory_report"])
        f.write("\n\n")

        f.write(f"Subcategory class count (clean): "
                f"{train_report['classes_subcategory_clean_count']}\n\n")

        # ---- Inference result ----
        f.write("=== INFERENCE RESULT ===\n")
        f.write(json.dumps(inference_result, indent=2, ensure_ascii=False))
        f.write("\n\n")

        f.write("===== END OF REPORT =====\n")


# -----------------------------
# Demo main
# -----------------------------
if __name__ == "__main__":
    CSV_PATH = "data/raw/911_recordings/metadata/911.csv"
    OUTPUT_TXT = "analysis_output.txt"

    models, train_report = train_models_from_911_csv(
        CSV_PATH,
        max_rows=200000,
        subcategory_min_count=5
    )

    print("=== TRAIN REPORT (Category) ===")
    print(train_report["category_report"])
    print("\n=== TRAIN REPORT (Subcategory, rare->OTHER) ===")
    print(train_report["subcategory_report"])
    print("\nSubcategory classes (clean) count:",
          train_report["classes_subcategory_clean_count"])

    sample_transcript = """
    Hello, please help! My father is unconscious and not breathing properly.
    We are at 123 Main Street, near the highway I-95. I think it's a heart attack.
    There are two people injured. Hurry!
    """

    inference_result = analyze_call_text(models, sample_transcript)

    print("\n=== INFERENCE RESULT ===")
    print(json.dumps(inference_result, indent=2, ensure_ascii=False))

    # ---- WRITE TO TXT ----
    write_output_to_txt(
        OUTPUT_TXT,
        train_report,
        inference_result
    )

    print(f"\nOutput written to: {OUTPUT_TXT}")



ModuleNotFoundError: No module named 'textblob'